In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import cv2
from pathlib import Path

In [2]:
!pip install pandas
!pip install matplotlib
!pip install opencv-python
!pip install tqdm

In [4]:
from pathlib import Path

output_path = Path("./Ar_Dataset")

for split in ["train", "val"]:
    (output_path / "images" / split).mkdir(parents=True, exist_ok=True)
    (output_path / "labels" / split).mkdir(parents=True, exist_ok=True)

In [2]:
!pip install ultralytics

  Using cached polars-1.39.3-py3-none-any.whl.metadata (10 kB)
  Using cached ultralytics_thop-2.0.18-py3-none-any.whl.metadata (14 kB)
  Using cached polars_runtime_32-1.39.3-cp310-abi3-win_amd64.whl.metadata (1.5 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
  Using cached filelock-3.25.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   -------- ------------------------------- 0.3/1.2 MB ? eta -:--:--
   ----------------- ---------------------- 0.5/1.2 MB 1.7 MB/s eta 0:00:01
   ------------------------- -------------- 0.8/1.2 MB 1.2 MB/s eta 0:00:01
   ---------------------------------- ----- 1

In [2]:
import os
import shutil
import random

# ── Configuration ────────────────────────────────────────────────────────────
SOURCE_DIR  = r"C:\Users\Mohamed Walid\Desktop\Internship\Code\Ar Dataset"
OUTPUT_DIR  = r"C:\Users\Mohamed Walid\Desktop\Internship\Code\Ar_Dataset_Split"
TRAIN_RATIO = 0.8
SEED        = 42
# ─────────────────────────────────────────────────────────────────────────────

def main():
    random.seed(SEED)

    # Collect all image files from the source folder
    image_extensions = {".jpg", ".jpeg", ".png", ".bmp"}
    all_images = [
        f for f in os.listdir(SOURCE_DIR)
        if os.path.splitext(f)[1].lower() in image_extensions
    ]

    if not all_images:
        print("❌  No image files found in the source directory.")
        return

    # Shuffle and split
    random.shuffle(all_images)
    split_idx   = int(len(all_images) * TRAIN_RATIO)
    train_files = all_images[:split_idx]
    val_files   = all_images[split_idx:]

    # Create output folders
    for split in ("train", "val"):
        for kind in ("images", "labels"):
            os.makedirs(os.path.join(OUTPUT_DIR, kind, split), exist_ok=True)

    def copy_pair(filename, split):
        """Copy an image + its matching .txt label to the right split folder."""
        stem      = os.path.splitext(filename)[0]
        label_file = stem + ".txt"

        src_img   = os.path.join(SOURCE_DIR, filename)
        src_lbl   = os.path.join(SOURCE_DIR, label_file)
        dst_img   = os.path.join(OUTPUT_DIR, "images", split, filename)
        dst_lbl   = os.path.join(OUTPUT_DIR, "labels", split, label_file)

        shutil.copy2(src_img, dst_img)

        if os.path.exists(src_lbl):
            shutil.copy2(src_lbl, dst_lbl)
        else:
            print(f"  ⚠️  No label found for {filename} — image copied without label.")

    # Copy files
    print(f"\n📂  Source : {SOURCE_DIR}")
    print(f"📂  Output : {OUTPUT_DIR}")
    print(f"\n📊  Total images : {len(all_images)}")
    print(f"   ✅  Train       : {len(train_files)}  ({TRAIN_RATIO*100:.0f}%)")
    print(f"   ✅  Val         : {len(val_files)}   ({(1-TRAIN_RATIO)*100:.0f}%)\n")

    for f in train_files:
        copy_pair(f, "train")
    for f in val_files:
        copy_pair(f, "val")

    print("✅  Dataset split complete!\n")
    print("Output structure:")
    print(f"""
{OUTPUT_DIR}
├── images/
│   ├── train/   ({len(train_files)} images)
│   └── val/     ({len(val_files)} images)
└── labels/
    ├── train/   ({len(train_files)} labels)
    └── val/     ({len(val_files)} labels)
""")

if __name__ == "__main__":
    main()


📂  Source : C:\Users\Mohamed Walid\Desktop\Internship\Code\Ar Dataset
📂  Output : C:\Users\Mohamed Walid\Desktop\Internship\Code\Ar_Dataset_Split

📊  Total images : 1596
   ✅  Train       : 1276  (80%)
   ✅  Val         : 320   (20%)

✅  Dataset split complete!

Output structure:

C:\Users\Mohamed Walid\Desktop\Internship\Code\Ar_Dataset_Split
├── images/
│   ├── train/   (1276 images)
│   └── val/     (320 images)
└── labels/
    ├── train/   (1276 labels)
    └── val/     (320 labels)



In [3]:
"""
Rename all images + matching .txt labels to clean sequential names
==================================================================
Before:
    IMG_0001.jpg, IMG_0001 (2).jpg, IMG_0001 (3).jpg ...
After:
    0001.jpg, 0002.jpg, 0003.jpg ...

Works on all 4 splits: images/train, images/val, labels/train, labels/val
"""

import os
import re

# ── Configuration ─────────────────────────────────────────────────────────────
BASE_DIR = r"C:\Users\Mohamed Walid\Desktop\Internship\Code\Ar_Dataset_Split"
# ──────────────────────────────────────────────────────────────────────────────

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

def rename_split(split: str):
    img_dir = os.path.join(BASE_DIR, "images", split)
    lbl_dir = os.path.join(BASE_DIR, "labels", split)

    if not os.path.exists(img_dir):
        print(f"  ⚠️  Skipping '{split}' — folder not found: {img_dir}")
        return

    # Collect all image files and sort them so order is consistent
    images = sorted([
        f for f in os.listdir(img_dir)
        if os.path.splitext(f)[1].lower() in IMAGE_EXTS
    ])

    print(f"\n📁  [{split}] Found {len(images)} images")

    renamed_imgs  = 0
    renamed_lbls  = 0
    missing_lbls  = 0

    for idx, old_img_name in enumerate(images, start=1):
        ext          = os.path.splitext(old_img_name)[1].lower()
        new_name     = f"{idx:04d}"          # e.g. "0001"
        new_img_name = new_name + ext        # e.g. "0001.jpg"
        old_stem     = os.path.splitext(old_img_name)[0]  # e.g. "IMG_0001 (2)"
        new_lbl_name = new_name + ".txt"
        old_lbl_name = old_stem + ".txt"

        old_img_path = os.path.join(img_dir, old_img_name)
        new_img_path = os.path.join(img_dir, new_img_name)
        old_lbl_path = os.path.join(lbl_dir, old_lbl_name)
        new_lbl_path = os.path.join(lbl_dir, new_lbl_name)

        # Rename image
        if old_img_path != new_img_path:
            os.rename(old_img_path, new_img_path)
            renamed_imgs += 1

        # Rename matching label if it exists
        if os.path.exists(old_lbl_path):
            if old_lbl_path != new_lbl_path:
                os.rename(old_lbl_path, new_lbl_path)
                renamed_lbls += 1
        else:
            missing_lbls += 1

    print(f"  ✅  Images renamed : {renamed_imgs}")
    print(f"  ✅  Labels renamed : {renamed_lbls}")
    if missing_lbls:
        print(f"  ⚠️   Labels missing : {missing_lbls} (images renamed anyway)")


def main():
    print(f"📂  Base dir: {BASE_DIR}\n")
    print("🔄  Renaming train and val splits...\n")

    for split in ("train", "val"):
        rename_split(split)

    print("\n✅  All done! Files renamed to 0001, 0002, 0003 ...")


if __name__ == "__main__":
    main()

📂  Base dir: C:\Users\Mohamed Walid\Desktop\Internship\Code\Ar_Dataset_Split

🔄  Renaming train and val splits...


📁  [train] Found 1276 images
  ✅  Images renamed : 1276
  ✅  Labels renamed : 1276

📁  [val] Found 320 images
  ✅  Images renamed : 320
  ✅  Labels renamed : 320

✅  All done! Files renamed to 0001, 0002, 0003 ...
